# P103 — Aleatorización de dominio para transferir redes profundas de la simulación al mundo real

## 1. Título y paper

**Paper:** *Domain Randomization for Transferring Deep Neural Networks from Simulation to the Real World*  
**Autoría:** Josh Tobin, Rachel Fong, Alex Ray, Jonas Schneider, Wojciech Zaremba, Pieter Abbeel  
**Año y venue:** 2017 · IROS 2017, 23–30 · arXiv:1703.06907  
**Nivel:** L2 · **Motor:** `domain_randomization`  
**Ficha completa:** [`P103_domain_randomization`](../../papers/foundational/P103_domain_randomization/README.md)

**Hito:** Invierte el objetivo del simulador: en vez de buscar fidelidad, busca que la realidad sea una variación más dentro del rango de entrenamiento.

- [arXiv:1703.06907](https://arxiv.org/abs/1703.06907)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Entrenar en simulación es barato y seguro; desplegar en el mundo real falla. El hueco entre simulación y realidad se atacaba mejorando el simulador, una carrera cara y sin final: siempre queda algo que no se modeló.
2. Ejecutar una implementación mínima de la propuesta: Aleatorizar agresivamente los parámetros del simulador —texturas, iluminación, posiciones de cámara, ruido— durante el entrenamiento. Si la variabilidad es suficiente, al modelo la realidad le parece una configuración más de las que ya vio.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P102
- P04


## 4. Intuición

El hueco entre simulación y realidad se atacaba mejorando el simulador: más física, mejores texturas, más fidelidad. Es una carrera sin final, porque siempre queda algo que nadie modeló. Tobin y sus coautores le dan la vuelta: si el simulador varía lo suficiente, la realidad es una variación más.


## 5. Concepto mínimo

```text
Enfoque clásico:   simulador ≈ realidad          ← perseguir fidelidad
Aleatorización:    realidad ∈ rango(simulador)   ← perseguir COBERTURA

En cada episodio se sortean texturas, luces, posiciones de cámara, ruido…
El modelo no puede depender de ninguna de esas cosas para resolver la tarea.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('domain_randomization', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué error tiene el modelo del simulador fijo en su propio simulador? ¿Y en la realidad?
2. ¿Y el modelo entrenado con parámetros aleatorizados?
3. ¿Cuál es el precio?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('domain_randomization', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('domain_randomization', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El modelo del simulador fijo tiene un error de **0,0501** en su propio simulador y de **0,8275** en la realidad. El aleatorizado tiene un error **mayor** en su simulador (0,6828) y **menor** en la realidad (0,5292).


## 10. Comentario pedagógico

Ese es el compromiso, y conviene decirlo entero: se cambia rendimiento en el caso nominal por robustez ante el caso desconocido. Un modelo aleatorizado es peor en cualquier configuración concreta. Solo compensa si de verdad no se sabe en qué configuración se va a desplegar — que es la situación normal.


## 11. Error o anti-patrón deliberado

Anti-patrón: aleatorizar y dar por resuelto el hueco.


In [ ]:
print('La aleatorizacion solo ayuda si la realidad cae DENTRO del rango que se sorteo.')
print('Si hay un fenomeno que nadie penso en aleatorizar, no aporta nada.')
print('Y aleatorizar de mas hace la tarea imposible de aprender: hay un equilibrio y no hay receta.')

## 12. Corrección

El compromiso, con los cuatro números delante:


In [ ]:
r = run_paper_lab('domain_randomization', seed=7)['result']
print('sim fijo   -> en su sim:', r['error_sim_fijo_en_su_propio_simulador'],
      '| en la realidad:', r['error_sim_fijo_en_la_realidad'])
print('aleatorio  -> en su sim:', r['error_aleatorizado_en_su_simulador'],
      '| en la realidad:', r['error_aleatorizado_en_la_realidad'])

## 13. Desafío guiado

Explica por qué el modelo aleatorizado es peor en su propio simulador, y por qué eso es esperable y no un defecto.


In [ ]:
r = run_paper_lab('domain_randomization', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena un modelo sobre datos sintéticos con un solo conjunto de condiciones y otro con condiciones aleatorizadas. Evalúa ambos sobre datos reales y documenta el compromiso.


## 15. Evidencia de aprendizaje

Guarda la tabla de los cuatro errores y tu enunciado del compromiso entre caso nominal y robustez.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P103_domain_randomization/README.md) · evaluación formal: [`assessments/papers/P103_domain_randomization.md`](../../assessments/papers/P103_domain_randomization.md)


## 16. Cierre

El robot ya puede aprender en simulación y funcionar fuera. Queda el otro cuerpo que un agente puede tener: la pantalla de otra persona.


## 17. Conexión con el siguiente hito

- P106

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
